# Analyse Bayesienne 

## Ajout des bibliothèques nécessaires

In [1]:
#importation des données
import os
import glob
import random

# Manipulation des données
import pandas as pd
import numpy as np

# Outils de Machine Learning (ici je vais utiliser Scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

print("Toutes les bibliothèques sont chargées avec succès !")

Toutes les bibliothèques sont chargées avec succès !


## Segmentation des fichiers en paquets de 5 phrases

In [ ]:
def segmenter_en_paquets(texte, taille_paquet=5):
    phrases = texte.splitlines()
    phrases = [p.strip() for p in phrases if p.strip()]

    paquets = []
    for i in range(0, len(phrases), taille_paquet):
        paquet = " ".join(phrases[i:i+taille_paquet])
        paquets.append(paquet)
    return paquets

chemin_dossier = "Z_vs_N"

liste_fichiers = sorted(glob.glob(os.path.join(chemin_dossier, "*.txt")))
print(f"Trouvé {len(liste_fichiers)} fichiers.")


random.seed(1)
random.shuffle(liste_fichiers)
liste_train = liste_fichiers[:40]
liste_test = liste_fichiers[40:]

print(f"Fichiers train : {len(liste_train)}")
print(f"Fichiers test  : {len(liste_test)}")


def construire_dataframe(liste_fichiers):

    donnees = []

    for chemin_fichier in liste_fichiers:
        nom_fichier = os.path.basename(chemin_fichier)
        if nom_fichier.endswith("clean.txt"):
            label = "Zola"
        else:
            label = "naturaliste"
        with open(chemin_fichier, "r", encoding="utf-8") as f:
            texte = f.read()
        blocs = segmenter_en_paquets(texte)
        for bloc in blocs:
            donnees.append({
                "texte": bloc,
                "label": label,
                "source": nom_fichier
            })
    return pd.DataFrame(donnees)


df_train = construire_dataframe(liste_train)
df_test = construire_dataframe(liste_test)

print("-"*30)
print("TRAIN")
print(df_train["label"].value_counts())

print("-"*30)
print("TEST")
print(df_test["label"].value_counts())

Trouvé 49 fichiers.
Fichiers train : 40
Fichiers test  : 9
------------------------------
TRAIN
label
Zola           22094
naturaliste    15301
Name: count, dtype: int64
------------------------------
TEST
label
Zola           5711
naturaliste    3037
Name: count, dtype: int64


In [4]:
print("Textes dans le train")
print(len(df_train))

sources_par_label = (
    df_train.groupby("label")["source"]
      .apply(lambda x: x.dropna().unique().tolist())
)

print("-" * 30)
print("Sources par label :")

for label, sources in sources_par_label.items():
    print(f"\n{label} :")
    for source in sources:
        print(f"  - {source}")
        
print("\n\n\nTextes dans le test)")
print(len(df_test))
        
sources_par_label = (
    df_test.groupby("label")["source"]
      .apply(lambda x: x.dropna().unique().tolist())
)

print("-" * 30)
print("Sources par label :")

for label, sources in sources_par_label.items():
    print(f"\n{label} :")
    for source in sources:
        print(f"  - {source}")

Textes dans le train
37395
------------------------------
Sources par label :

Zola :
  - 1873_3_Le_ventre_de_Paris._clean.txt
  - 1888_16_Le_reve._clean.txt
  - 1882_10_Pot-bouille._clean.txt
  - 1876_6_Son_Excellence_Eugene_Rougon._clean.txt
  - 1887_15_La_terre._clean.txt
  - 1884_12_La_joie_de_vivre._clean.txt
  - 1893_20_Le_docteur_Pascal._clean.txt
  - 1885_13_Germinal._clean.txt
  - 1883_11_Au_Bonheur_des_dames._clean.txt
  - 1874_4_La_conquete_de_Plassans._clean.txt
  - 1892_19_La_debacle._clean.txt
  - 1891_18_L_argent._clean.txt
  - 1871_1_La_fortune_des_Rougon._clean.txt
  - 1871_2_La_curee._clean.txt
  - 1877_7_L_assommoir._clean.txt
  - 1886_14_L_oeuvre._clean.txt

naturaliste :
  - malheur d'Henriette Gérard.txt
  - Amis de la nature.txt
  - En route.txt
  - rois_en_exil.txt
  - le_Nabab.txt
  - Germinie Lacerteux.txt
  - L'education_sentimentale.txt
  - Confessions de Sylvius.txt
  - soutien_de_famille.txt
  - Paule Méré.txt
  - Mont Oriol.txt
  - Barbier de Paris.txt
  

In [32]:
df_train.head()


,texte,label,source
0,"Au milieu du grand silence, et dans le désert ...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
1,"Il marchait, dormant à demi, dodelinant des or...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
2,"cria un des hommes, qui s’était mis à genoux s...",Zola,1873_3_Le_ventre_de_Paris._clean.txt
3,Eh! l’homme! dit-elle doucement. Mais les char...,Zola,1873_3_Le_ventre_de_Paris._clean.txt
4,"Il regardait Mme François d’un air effaré, san...",Zola,1873_3_Le_ventre_de_Paris._clean.txt


In [33]:
df_test.head()

,texte,label,source
0,I – Un article ?… Tu me demandes s’il y a un a...,naturaliste,Charles Demailly.txt
1,"– Après, après ? tu poses ta femme : une coméd...",naturaliste,Charles Demailly.txt
2,L’autre était un jeune homme de trente-quatre ...,naturaliste,Charles Demailly.txt
3,"Le second, tout entier à faire des appels au m...",naturaliste,Charles Demailly.txt
4,"Ces cinq hommes, qui avaient l’oreille à la bo...",naturaliste,Charles Demailly.txt


## Début d'entrainement du modèle Bayesien

In [5]:
X_train = df_train["texte"]
y_train = df_train["label"]

X_test = df_test["texte"]
y_test = df_test["label"]

modele_bayes = make_pipeline(
    TfidfVectorizer(
        max_features=10000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9),
    MultinomialNB(alpha=1.0)
)

modele_bayes.fit(X_train, y_train)

print("Modèle entraîné.")

Modèle entraîné.


In [6]:
predictions = modele_bayes.predict(X_test)

print(classification_report(y_test, predictions))

cm = confusion_matrix(y_test, predictions)
print(cm)

              precision    recall  f1-score   support

        Zola       0.94      0.94      0.94      5711
 naturaliste       0.89      0.89      0.89      3037

    accuracy                           0.92      8748
   macro avg       0.91      0.91      0.91      8748
weighted avg       0.92      0.92      0.92      8748

[[5375  336]
 [ 346 2691]]


In [7]:
vectorizer = modele_bayes.named_steps["tfidfvectorizer"]
classifieur = modele_bayes.named_steps["multinomialnb"]

mots = vectorizer.get_feature_names_out()
classes = classifieur.classes_

print("Ordre des classes :", classes)

# Différence des log-probabilités entre les deux classes
score_discriminant = (
    classifieur.feature_log_prob_[0]
    - classifieur.feature_log_prob_[1]
)

resultats_mots = pd.DataFrame({
    "mot": mots,
    f"score_{classes[0]}": score_discriminant
})

# Mots les plus caractéristiques de la première classe
mots_classe_0 = resultats_mots.sort_values(
    f"score_{classes[0]}",
    ascending=False
).head(30)

# Mots les plus caractéristiques de la deuxième classe
mots_classe_1 = resultats_mots.sort_values(
    f"score_{classes[0]}",
    ascending=True
).head(30)

print(f"\nMots les plus caractéristiques de {classes[0]} :")
for _, ligne in mots_classe_0.iterrows():
    print(f"{ligne['mot']:<25} {ligne[f'score_{classes[0]}']:.3f}")

print(f"\nMots les plus caractéristiques de {classes[1]} :")
for _, ligne in mots_classe_1.iterrows():
    print(f"{ligne['mot']:<25} {-ligne[f'score_{classes[0]}']:.3f}")

Ordre des classes : ['Zola' 'naturaliste']

Mots les plus caractéristiques de Zola :
rougon                    4.506
mouret                    4.150
gervaise                  4.106
saccard                   4.067
pauline                   3.977
coupeau                   3.965
denise                    3.799
étienne                   3.757
octave                    3.737
buteau                    3.733
florent                   3.623
faujas                    3.621
chanteau                  3.535
plassans                  3.521
lisa                      3.503
josserand                 3.466
claude                    3.420
renée                     3.413
abbé faujas               3.359
maurice                   3.357
silvère                   3.355
clorinde                  3.295
maheu                     3.250
lantier                   3.243
lorilleux                 3.215
fouan                     3.212
macquart                  3.191
sandoz                    3.181
duveyrier          

In [37]:
for mot in ["gervaise", "étienne", "denise", "pauline", "coupeau"]:
    nb = df_test["texte"].str.contains(mot, case=False).sum()
    print(mot, nb)

gervaise 1
étienne 5
denise 1
pauline 82
coupeau 0


In [38]:
# Récupération des étapes du pipeline
vectorizer = modele_bayes.named_steps["tfidfvectorizer"]
classifieur = modele_bayes.named_steps["multinomialnb"]

# Vocabulaire appris sur le train
termes = vectorizer.get_feature_names_out()

# Ordre réel des classes
classes = classifieur.classes_
print("Ordre des classes :", classes)

# Transformation du jeu de test avec le vectorizer déjà entraîné
X_test_tfidf = vectorizer.transform(df_test["texte"])

# Nombre d'extraits du test contenant chaque terme
presence_test = np.asarray((X_test_tfidf > 0).sum(axis=0)).ravel()

# Somme des poids TF-IDF de chaque terme dans le test
poids_tfidf_test = np.asarray(X_test_tfidf.sum(axis=0)).ravel()

# Score discriminant appris par Naive Bayes
# score positif : classe 0
# score négatif : classe 1
score_discriminant = (
    classifieur.feature_log_prob_[0]
    - classifieur.feature_log_prob_[1]
)

resultats = pd.DataFrame({
    "terme": termes,
    "score_discriminant": score_discriminant,
    "nb_extraits_test": presence_test,
    "poids_tfidf_test": poids_tfidf_test
})

# On conserve uniquement les termes présents dans le test
resultats_test = resultats[
    resultats["nb_extraits_test"] > 0
].copy()

print(f"\nNombre de termes du vocabulaire présents dans le test : "
      f"{len(resultats_test)}")

Ordre des classes : ['Zola' 'naturaliste']

Nombre de termes du vocabulaire présents dans le test : 9705


In [27]:
n = 30

termes_classe_0 = (
    resultats_test
    .sort_values("score_discriminant", ascending=False)
    .head(n)
)

termes_classe_1 = (
    resultats_test
    .sort_values("score_discriminant", ascending=True)
    .head(n)
)

print(f"\nTermes présents dans le test les plus caractéristiques de {classes[0]} :")

for _, ligne in termes_classe_0.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"score={ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )

print(f"\nTermes présents dans le test les plus caractéristiques de {classes[1]} :")

for _, ligne in termes_classe_1.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"score={-ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )


Termes présents dans le test les plus caractéristiques de Zola :
rougon                    score=5.431  extraits=3
gervaise                  score=5.037  extraits=1
mouret                    score=4.924  extraits=171
pauline                   score=4.810  extraits=84
denise                    score=4.731  extraits=1
octave                    score=4.602  extraits=3
étienne                   score=4.555  extraits=1
lisa                      score=4.347  extraits=15
plassans                  score=4.304  extraits=15
macquart                  score=4.049  extraits=1
lantier                   score=4.041  extraits=10
claude                    score=3.935  extraits=2
boche                     score=3.874  extraits=1
maurice                   score=3.853  extraits=3
miette                    score=3.682  extraits=3
nana                      score=3.636  extraits=848
baudu                     score=3.627  extraits=1
prussiens                 score=3.575  extraits=4
caroline                  

In [40]:
resultats_test["contribution_test"] = (
    resultats_test["score_discriminant"]
    * resultats_test["poids_tfidf_test"]
)

In [41]:
contributions_zola = (
    resultats_test
    .sort_values("contribution_test", ascending=False)
    .head(30)
)

contributions_naturalistes = (
    resultats_test
    .sort_values("contribution_test", ascending=True)
    .head(30)
)

print(f"\nTermes ayant le plus contribué aux prédictions {classes[0]} :")

for _, ligne in contributions_zola.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"contribution={ligne['contribution_test']:.3f}  "
        f"score={ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )

print(f"\nTermes ayant le plus contribué aux prédictions {classes[1]} :")

for _, ligne in contributions_naturalistes.iterrows():
    print(
        f"{ligne['terme']:<25} "
        f"contribution={-ligne['contribution_test']:.3f}  "
        f"score={-ligne['score_discriminant']:.3f}  "
        f"extraits={int(ligne['nb_extraits_test'])}"
    )


Termes ayant le plus contribué aux prédictions Zola :
nana                      contribution=369.117  score=2.857  extraits=703
qu                        contribution=241.057  score=1.253  extraits=3151
elle                      contribution=183.320  score=0.392  extraits=4834
était                     contribution=136.421  score=0.724  extraits=3571
eugène                    contribution=115.724  score=1.943  extraits=323
qu il                     contribution=114.025  score=1.252  extraits=1388
qu elle                   contribution=112.389  score=1.382  extraits=1024
avait                     contribution=104.201  score=0.589  extraits=3319
ça                        contribution=98.903  score=0.983  extraits=1139
il                        contribution=87.976  score=0.211  extraits=6013
lorsqu                    contribution=86.574  score=2.489  extraits=399
abbé                      contribution=81.347  score=2.555  extraits=269
mouret                    contribution=78.060  score=